In [1]:
from ocpmodels.trainers import ForcesTrainer
from ocpmodels.trainers import EnergyTrainer
from ocpmodels.datasets import LmdbDataset
from ocpmodels import models
from ocpmodels.common import logger
from ocpmodels.common.utils import setup_logging
setup_logging()

import numpy as np
import copy
import os
import logging
import os
import torch
from torch_geometric.data import Data
from ocpmodels.models.gemnet_oc.gemnet_oc import GemNetOC 


In [2]:
train_src = "/home/zjy/code/mycode/ocp/test_data/test_1"
val_src = "/home/zjy/code/mycode/ocp/test_data/test_1"
# train_src = "/home/ubuntu/dataset/gemnet_test/lmdb"
# val_src = "/home/ubuntu/dataset/gemnet_test/lmdb"
# train_src = "/home/ubuntu/code/opencat/ocp/test_data/lmdb_id_H"
# val_src = "/home/ubuntu/code/opencat/ocp/test_data/lmdb_id_H"

In [3]:
train_dataset = LmdbDataset({"src": train_src})

energies = []
for data in train_dataset:
  energies.append(data.y)

mean = np.mean(energies)
stdev = np.std(energies)
train_dataset

In [4]:
task = {
    'dataset': 'trajectory_lmdb', # dataset used for the S2EF task
    'description': 'Regressing to energies and forces for DFT trajectories from OCP',
    'type': 'regression',
    'metric': 'mae',
    'labels': ['potential energy'],
    'grad_input': 'atomic forces',
    'train_on_free_atoms': True,
    'eval_on_free_atoms': True
}
# Model
model = {
    'name': 'gemnet_oc',
    "num_spherical": 7,
    "num_radial": 128,
    "num_blocks": 4,
    "emb_size_atom": 256,
    "emb_size_edge": 512,
    "emb_size_trip_in": 64,
    "emb_size_trip_out": 64,
    "emb_size_quad_in": 32,
    "emb_size_quad_out": 32,
    "emb_size_aint_in": 64,
    "emb_size_aint_out": 64,
    "emb_size_rbf": 16,
    "emb_size_cbf": 16,
    "emb_size_sbf": 32,
    "num_before_skip": 2,
    "num_after_skip": 2,
    "num_concat": 1,
    "num_atom": 3,
    "num_output_afteratom": 3,
    "cutoff": 5.0,
    "cutoff_qint": 12.0,
    "cutoff_aeaint": 12.0,
    "cutoff_aint": 12.0,
    "max_neighbors": 8,
    "max_neighbors_qint": 8,
    "max_neighbors_aeaint": 20,
    "max_neighbors_aint": 1000,
    "rbf": {"name": "gaussian"},
    "envelope": {"name": "polynomial", "exponent": 5},
    "cbf": {"name": "spherical_harmonics"},
    "sbf": {"name": "legendre_outer"},
    "extensive": True,
    "output_init": "HeOrthogonal",
    "activation": "silu",
    "scale_file": "configs/s2ef/all/gemnet/scaling_factors/gemnet-oc.pt",
    "regress_forces": True,
    "direct_forces": True,
    "forces_coupled": False,
    "quad_interaction": True,
    "atom_edge_interaction": True,
    "edge_atom_interaction": True,
    "atom_interaction": True,
    "num_atom_emb_layers": 2,
    "num_global_out_layers": 2,
    "qint_tags": [0, 1]
}
# Optimizer
optimizer = {
    'batch_size': 2,         # originally 32
    'eval_batch_size': 2,    # originally 32
    'load_balancing': 'atoms',
    'num_workers': 2,
    'lr_initial': 5.e-4,
    'optimizer': 'AdamW',
    'optimizer_params': {"amsgrad": True},
    'scheduler': "ReduceLROnPlateau",
    'mode': "min",
    'factor': 0.8,
    'patience': 3,
    'max_epochs': 1,         # used for demonstration purposes
    'force_coefficient': 100,
    'ema_decay': 0.999,
    'clip_grad_norm': 10,
    'loss_energy': 'mae',
    'loss_force': 'l2mae',
}
# Dataset
dataset = [
  {'src': train_src,
   'normalize_labels': True,
   "target_mean": mean,
   "target_std": stdev,
   "grad_target_mean": 0.0,
   "grad_target_std": stdev
   }, # train set 
  {'src': val_src}, # val set (optional)
]

In [5]:
trainer = EnergyTrainer(
    task=task,
    model=copy.deepcopy(model), # copied for later use, not necessary in practice.
    dataset=dataset,
    optimizer=optimizer,
    identifier="S2EF-example",
    run_dir="./", # directory to save results if is_debug=False. Prediction files are saved here so be careful not to override!
    is_debug=False, # if True, do not save checkpoint, logs, or results
    is_hpo=False,
    print_every=5,
    seed=0, # random seed to use
    logger="tensorboard", # logger of choice (tensorboard and wandb supported)
    local_rank=0,
    amp=True, # use PyTorch Automatic Mixed Precision (faster training and less memory usage),
)

amp: true
cmd:
  checkpoint_dir: ./checkpoints/2024-07-16-14-49-36-S2EF-example
  commit: null
  identifier: S2EF-example
  logs_dir: ./logs/tensorboard/2024-07-16-14-49-36-S2EF-example
  print_every: 5
  results_dir: ./results/2024-07-16-14-49-36-S2EF-example
  seed: 0
  timestamp_id: 2024-07-16-14-49-36-S2EF-example
dataset:
  grad_target_mean: 0.0
  grad_target_std: !!python/object/apply:numpy.core.multiarray.scalar
  - &id001 !!python/object/apply:numpy.dtype
    args:
    - f8
    - false
    - true
    state: !!python/tuple
    - 3
    - <
    - null
    - null
    - null
    - -1
    - -1
    - 0
  - !!binary |
    14v5cU+E9j8=
  normalize_labels: true
  src: /home/zjy/code/mycode/ocp/test_data/test_1
  target_mean: !!python/object/apply:numpy.core.multiarray.scalar
  - *id001
  - !!binary |
    7QKhX9XZA8A=
  target_std: !!python/object/apply:numpy.core.multiarray.scalar
  - *id001
  - !!binary |
    14v5cU+E9j8=
gpus: 1
logger: tensorboard
model: gemnet_oc
model_attributes:
  

fatal: Not a valid object name HEAD


2024-07-16 14:48:38 (INFO): Loaded GemNetOC with 38886113 parameters.


2024-07-16 14:48:38 (WARNING): Model gradient logging to tensorboard not yet supported.


In [6]:
trainer.train()

energy_mae: 2.28e+00, energy_mse: 5.27e+00, energy_within_threshold: 0.00e+00, loss: 1.62e+00, lr: 5.00e-04, epoch: 1.00e+00, step: 5.00e+00
2024-07-16 14:48:44 (INFO): Evaluating on val.


device 0: 100%|██████████| 5/5 [00:00<00:00, 12.17it/s]

2024-07-16 14:48:44 (INFO): energy_mae: 1.2246, energy_mse: 1.9809, energy_within_threshold: 0.0000, loss: 0.8701, epoch: 1.0000
